In [1]:
import json
import pandas as pd
import re

d:\anaconda3\lib\site-packages\pandas\core\computation\expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [43]:
with open("export_145865_project-145865-at-2025-05-04-19-12-8471309f_without_history.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [3]:
data[0].keys()

dict_keys(['id', 'annotations', 'file_upload', 'drafts', 'predictions', 'agreement', 'data', 'meta', 'created_at', 'updated_at', 'inner_id', 'total_annotations', 'cancelled_annotations', 'total_predictions', 'comment_count', 'unresolved_comment_count', 'last_comment_updated_at', 'project', 'updated_by', 'comment_authors'])

In [17]:
data[0]['agreement'], data[0]['total_predictions'], data[0]['meta'], data[0]['total_annotations']


(50.0, 2, {}, 4)

In [18]:
len(data[0]['annotations'])

4

In [34]:
data[0]['annotations'][3]['result'][0]['value']['choices'][0]

'Good'

In [52]:
selected_data = []

for item in data:
    annots = []
    for i in range(item['total_annotations']):
        try:
            annot = item['annotations'][i]['result'][0]['value']['choices'][0]
            annots.append(annot)
        except:
            # print("item['total_annotations']", item['total_annotations'])
            # print("item['annotations']", item['annotations'])
            # print()
            # has result = []
            continue
    selected_data.append(
        {"extracted_content": item['data'].get("extracted_content", ""),
        "total_annotations": len(annots),
        # data[0]['annotations'][0]['result'][0]['value']['choices'][0] is 'Good' or 'Bad'
        "Annotations": annots,
        })

# Step 3: Convert to DataFrame
df = pd.DataFrame(selected_data)
df

,extracted_content,total_annotations,Annotations
0,"{'title': 'نبرد مورچه‌خورت و آزادسازی اصفهان',...",4,"[Good, Bad, Good, Good]"
1,{'title': 'اردوکشی نادرشاه به میان‌رودان و قفق...,3,"[Good, Good, Good]"
2,"{'title': 'حمله نادرشاه به سند و هند', 'causes...",3,"[Good, Good, Bad]"
3,"{'title': '', 'causes': [], 'impact': {'legal'...",3,"[Bad, Bad, Bad]"
4,"{'title': 'اشغال ایران در جنگ جهانی اول', 'cau...",3,"[Good, Good, Good]"
...,...,...,...
445,{'title': 'جنگ‌های ایران و یونان (قرن پنجم قبل...,3,"[Good, Good, Good]"
446,"{'title': 'انقلاب اسلامی ایران', 'causes': ['ا...",3,"[Good, Good, Good]"
447,"{'title': 'انقلاب اسلامی ایران', 'causes': ['ا...",3,"[Good, Good, Good]"
448,"{'title': 'انقلاب اسلامی ایران', 'causes': ['ا...",3,"[Good, Good, Good]"


In [ ]:
# df has a column "Annotations" with values like ['Good', 'Bad', 'Good', 'Bad']
# I want to keep the rows with more that 50% of Good values in the Annotations and filter out others
filtered_data = df[df['Annotations'].apply(lambda x: (x.count('Good') / len(x)) > 0.5)]
filtered_data

,extracted_content,total_annotations,Annotations
0,"{'title': 'نبرد مورچه‌خورت و آزادسازی اصفهان',...",4,"[Good, Bad, Good, Good]"
1,{'title': 'اردوکشی نادرشاه به میان‌رودان و قفق...,3,"[Good, Good, Good]"
2,"{'title': 'حمله نادرشاه به سند و هند', 'causes...",3,"[Good, Good, Bad]"
4,"{'title': 'اشغال ایران در جنگ جهانی اول', 'cau...",3,"[Good, Good, Good]"
5,"{'title': 'اشغال ایران در جنگ جهانی دوم', 'cau...",3,"[Good, Good, Good]"
...,...,...,...
445,{'title': 'جنگ‌های ایران و یونان (قرن پنجم قبل...,3,"[Good, Good, Good]"
446,"{'title': 'انقلاب اسلامی ایران', 'causes': ['ا...",3,"[Good, Good, Good]"
447,"{'title': 'انقلاب اسلامی ایران', 'causes': ['ا...",3,"[Good, Good, Good]"
448,"{'title': 'انقلاب اسلامی ایران', 'causes': ['ا...",3,"[Good, Good, Good]"


In [54]:
# Now we save the filtered data to a new JSON file and only keep the extracted_content
extracted_contents = filtered_data['extracted_content']
extracted_contents

0      {'title': 'نبرد مورچه‌خورت و آزادسازی اصفهان',...
1      {'title': 'اردوکشی نادرشاه به میان‌رودان و قفق...
2      {'title': 'حمله نادرشاه به سند و هند', 'causes...
4      {'title': 'اشغال ایران در جنگ جهانی اول', 'cau...
5      {'title': 'اشغال ایران در جنگ جهانی دوم', 'cau...
                             ...                        
445    {'title': 'جنگ‌های ایران و یونان (قرن پنجم قبل...
446    {'title': 'انقلاب اسلامی ایران', 'causes': ['ا...
447    {'title': 'انقلاب اسلامی ایران', 'causes': ['ا...
448    {'title': 'انقلاب اسلامی ایران', 'causes': ['ا...
449    {'title': 'انقلاب اسلامی ایران', 'causes': ['ا...
Name: extracted_content, Length: 380, dtype: object

In [55]:

extracted_contents = extracted_contents.tolist()
# Save to a new JSON file
with open("filtered_extracted_contents.json", "w", encoding="utf-8") as f:
    json.dump(extracted_contents, f, ensure_ascii=False, indent=4)

# Save to a new CSV file
filtered_data.to_csv("filtered_extracted_contents.csv", index=False, encoding="utf-8-sig")